# Toxicité moléculaire avec GNN (version améliorée)

**Nom :** ............................................................  
**Prénom :** ........................................................  

Ce notebook montre un modèle GNN amélioré avec DeepChem.

In [14]:
import deepchem as dc
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

## Chargement dataset Tox21

In [15]:
# 1. On crée le convertisseur en graphes
featurizer = dc.feat.ConvMolFeaturizer()

# 2. On charge Tox21 en forçant l'utilisation de ce convertisseur
import os
local_cache = "./datasets/tox21_cache"
os.makedirs(local_cache, exist_ok=True)
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer, save_dir=local_cache, data_dir=local_cache)
train_dataset, valid_dataset, test_dataset = datasets


## Modèle GNN amélioré

In [16]:
model = dc.models.GraphConvModel(
    n_tasks=len(tasks),
    mode='classification',
    dropout=0.3,
    batch_size=64,
    learning_rate=5e-4
)

## Entraînement

In [17]:
model.fit(train_dataset, nb_epoch=50)

0.7143539428710938

## Évaluation

In [18]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print(model.evaluate(test_dataset, [metric]))

{'roc_auc_score': 0.6917324898311318}


## Test sur molécules

In [19]:
smiles_a_tester = {
    "Paracétamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Valdécoxib": "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N"
}

noms_molecules = list(smiles_a_tester.keys())
smiles = list(smiles_a_tester.values())

graphes_medocs = featurizer.featurize(smiles)
dataset = dc.data.NumpyDataset(X=graphes_medocs)

# Prédictions brutes
predictions = model.predict(dataset)

for i, nom in enumerate(noms_molecules):
    print(f"\n=== PROFIL DE TOXICITÉ : {nom} ===")
    
    # Pour chaque molécule, on regarde ses 12 scores
    for index, nom_cible in enumerate(tasks):
        # On regarde la probabilité d'être toxique (indice 1)
        probabilite = predictions[i][index][1] 
        
        if probabilite > 0.5:
            print(f"⚠️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
        else:
            print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")



=== PROFIL DE TOXICITÉ : Paracétamol ===
  Sûr    sur NR-AR      : 49.3%
⚠️ DANGER sur NR-AR-LBD  : 63.1%
⚠️ DANGER sur NR-AhR     : 79.7%
  Sûr    sur NR-Aromatase : 8.3%
⚠️ DANGER sur NR-ER      : 66.7%
⚠️ DANGER sur NR-ER-LBD  : 67.0%
  Sûr    sur NR-PPAR-gamma : 35.9%
⚠️ DANGER sur SR-ARE     : 59.0%
⚠️ DANGER sur SR-ATAD5   : 59.0%
  Sûr    sur SR-HSE     : 42.8%
⚠️ DANGER sur SR-MMP     : 55.8%
  Sûr    sur SR-p53     : 46.4%

=== PROFIL DE TOXICITÉ : Valdécoxib ===
⚠️ DANGER sur NR-AR      : 84.7%
⚠️ DANGER sur NR-AR-LBD  : 72.2%
⚠️ DANGER sur NR-AhR     : 70.6%
⚠️ DANGER sur NR-Aromatase : 89.1%
⚠️ DANGER sur NR-ER      : 53.4%
⚠️ DANGER sur NR-ER-LBD  : 66.4%
⚠️ DANGER sur NR-PPAR-gamma : 73.3%
⚠️ DANGER sur SR-ARE     : 60.5%
  Sûr    sur SR-ATAD5   : 24.1%
  Sûr    sur SR-HSE     : 37.1%
⚠️ DANGER sur SR-MMP     : 74.1%
  Sûr    sur SR-p53     : 23.4%


[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[08:02:14] DEPRECATIO

## Partie 4 : Bilan sur l'amélioration du modèle

### Questions :
1. **Hyperparamètres** : Dans ce modèle amélioré, nous avons ajusté le `dropout` à 0.3 et le `learning_rate` à 5e-4. À quoi sert la technique du *dropout* pendant l'entraînement d'un réseau de neurones ?
2. **Durée de l'entraînement** : Le paramètre `nb_epoch` est passé de 10 à 50. Quel est l'avantage de laisser le modèle s'entraîner plus longtemps, et quel est le risque principal si on l'entraîne trop (surapprentissage ou overfitting) ?
3. **Score et limites** : Le score ROC-AUC s'est amélioré (environ 0.697) par rapport au modèle de base. Pourquoi est-il si difficile d'obtenir un modèle qui prédit à 100% la toxicité d'une molécule sur un organisme vivant ?

*(Double-cliquez sur cette cellule pour rédiger vos réponses ci-dessous :)*

* **Réponse 1 :** ...
* **Réponse 2 :** ...
* **Réponse 3 :** ...
